# Script Outline

Import ACS 1 Year Estimates table and import counties layers from TIGER.  Merge these files together to make county level maps

- Prepare Workspace
- Import Data
- Data Cleaning
- Maps
- Exports

CRS Reprojection source
https://spatialreference.org/ref/epsg/2226/


## Prepare Workspace

#### Import packages

In [ ]:
# General
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import json


# Geographic
import geopandas as gpd
from census import Census
from us import states
import censusdata as acs
import pyproj

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

#### File paths

In [ ]:
# Working directory
path_projects = os.path.dirname(os.path.dirname(os.getcwd()))
print(path_projects)

# Set file paths
rootpath = os.path.join(path_projects, 'SACOG Exam')
path_in     = os.path.join(rootpath, 'Raw Data'     )
path_out    = os.path.join(rootpath, 'Python Output')
path_config = os.path.join(rootpath, 'config'       )

In [ ]:
# Supress scientific notation
pd.options.display.float_format = '{:.2f}'.format

## Import Data

In [ ]:
# Import county FIPS mapping
df_fips = pd.read_excel(os.path.join(path_out, 'County to FIPS Code Mapping.xlsx')
                        , dtype={'State FIPS': object, 'County FIPS': object})
df_fips.head()

In [ ]:
# Import ACS5 data
df_acs1 = pd.read_excel(os.path.join(path_out, 'Step 01a_Cleaned ACS1 Table B08006_2024-02-24.xlsx')
                      , dtype={'State FIPS': object, 'County FIPS': object, 'Tract ID': object})
df_acs1.head(6)

In [ ]:
# Access shapefile of US Counties
# initialize empty list to store geopandas dataframes
# set which county FIPS, years, and state to import
# import using TIGER url query
# set year
# concatenate

list_gpd_counties = []
fips_sac = df_acs1['County FIPS'].unique()
years = [2018, 2019, 2021, 2022]
for year in tqdm(years):
    temp = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER" + str(year) + "/COUNTY/tl_" + str(year) + "_us_county.zip")
    temp = temp[(temp['COUNTYFP'].isin(fips_sac)) & (temp['STATEFP'] == '06')]
    temp['Year'] = year
    list_gpd_counties.append(temp)
gpd_counties = pd.concat(list_gpd_counties)   


# Reproject shapefile to UTM Zone 17N
# Show
# Check shapefile projection

gpd_counties = gpd_counties.to_crs(epsg = 2226)
print(gpd_counties.head(2))
print('Shape: ', gpd_counties.shape)
print("\nThe shapefile projection is: {}".format(gpd_counties.crs))

## Data Cleaning

In [ ]:
df_acs1_merge = df_acs1.drop(columns = ['State FIPS']).rename(columns = {'County FIPS':'COUNTYFP'})
df_acs1_merge.head()

In [ ]:
# Join the attributes of the dataframes together
# Source: https://geopandas.org/docs/user_guide/mergingdata.html
gpd_counties_acs = gpd_counties.merge(df_acs1_merge, on = ["COUNTYFP", "Year"])
gpd_counties_acs = gpd_counties_acs[["COUNTYFP", "geometry", "Year", 'County Name'
                                 , "Population"
                                 , "Population commuting alone"
                                 , "Population working from home"]]

# Show
print(gpd_counties_acs.head(2))
print('Shape: ', gpd_counties_acs.shape)

## Maps

#### Counties

In [ ]:
# Estimate percentages
gpd_counties_acs["Population commuting alone (%)"  ] = gpd_counties_acs["Population commuting alone"  ]/gpd_counties_acs["Population"]
gpd_counties_acs["Population working from home (%)"] = gpd_counties_acs["Population working from home"]/gpd_counties_acs["Population"]

# Normalize metrics by population
gpd_counties_acs["Population commuting alone_per 1k pop"  ] = gpd_counties_acs["Population commuting alone"  ]/(gpd_counties_acs['Population']/1000)
gpd_counties_acs["Population working from home_per 1k pop"] = gpd_counties_acs["Population working from home"]/(gpd_counties_acs['Population']/1000)

# Normalize metrics by land area
gpd_counties_acs['acres'] = gpd_counties_acs['geometry'].area / 43560 # acres of each county
gpd_counties_acs["Population commuting alone_per 1k acres"  ] = gpd_counties_acs["Population commuting alone"  ]/(gpd_counties_acs['acres']/1000)
gpd_counties_acs["Population working from home_per 1k acres"] = gpd_counties_acs["Population working from home"]/(gpd_counties_acs['acres']/1000)

In [ ]:
# set metric to map
# Create subplots and plot data

metric = "Population working from home_per 1k pop"

fig, ax = plt.subplots(1, 1, figsize = (20, 10))

gpd_counties_acs[gpd_counties_acs['Year'] == 2021].plot(
    column = metric,
    ax = ax,
    cmap = "RdPu",
    legend = True)

plt.style.use('bmh')
ax.set_title('Population working from home (per 1,000 people)', fontdict = {'fontsize': '25', 'fontweight' : '3'})

# plt.savefig(os.path.join(path_out, 'Population working from home (per 1,000 people) 2021.png'))

In [ ]:
# # Organize specific dataframe and geojson object for mapping with px.choropleth
# # Create index field for mapping
# # Convert to 4326 CRS for mapping


# temp = gpd_counties_acs.copy()
# temp.reset_index(inplace = True)
# temp.rename(columns = {'index':'ID'}, inplace = True)

# # reading in the shapefile
# map_df = temp[['ID', 'geometry']]
# map_df.to_crs(pyproj.CRS.from_epsg(4326), inplace=True)

# df = temp.drop(columns = ['geometry'])

# # join the geodataframe with the cleaned up csv dataframe
# merged = map_df.set_index('ID').join(df.set_index('ID'))

In [ ]:
# # Iterate through every metric for mapping
# # Export maps as interactive plotly maps in .html files

# metrics = ["Population"
#           , "Population commuting alone"  
#           , "Population working from home"
#           , "Population commuting alone (%)"  
#           , "Population working from home (%)"
#           , "Population commuting alone_per 1k pop"  
#           , "Population working from home_per 1k pop"
#           , "Population commuting alone_per 1k acres"  
#           , "Population working from home_per 1k acres"]


# for metric in tqdm(metrics):
#     fig = px.choropleth(merged
#                         , geojson=merged.geometry
#                         , locations=merged.index
#                         , color=metric
#                         , color_continuous_scale="Blues"
#                         , animation_frame = 'Year')
#     fig.update_geos(fitbounds="locations", visible=False)
    
#     fig.write_html(
#         os.path.join(
#             path_out
#             , 'plotly'
#             , 'ACS1'
#             , 'maps'
#             , ''.join([metric
#                        , "_Counties_"
#                        , '_ACS1_'
#                        , date.today().strftime("%Y-%m-%d")
#                        , '.html'])
#         )
#     )

## Exports

In [ ]:
# df_acs1_counties = gpd_counties_acs.drop(columns = 'geometry')
# df_acs1_counties.head()

In [ ]:
# # Set output name
# name_output_counties = ['Step 02a_ACS1 Transportation Metrics_', 'Counties_'  , date.today().strftime("%Y-%m-%d"),'.xlsx']
# name_output_counties = "".join(name_output_counties)

In [ ]:
# Export to excel files
# df_acs1_counties.to_excel(os.path.join(path_out, name_output_counties), index=False)

In [ ]:
# Export to geojson files
# gpd_counties_acs.to_file(os.path.join(path_out, "gpd_counties_acs1.geojson"), driver="GeoJSON")